In [1]:
from modpath import modify_path

modify_path()

In [ ]:
import pathlib
from src import pipelines
from src.db_config import DatasetSplit
from src.model_config import ModelVersion, TARGET

In [3]:
FIGS_DIR = pathlib.Path.cwd().parent.joinpath('figs/02/')

# Model v2.0

Drop low SHAP features, replace 0 ratings with 3 ratings, and reformat `class` to only check for business class vs non-business class.

Removed: `age`, `departure_delay`, `flight_distance`, `food_drink`, `gate_location`, `online_booking_ease`, `class`

Added: `is_business_class`

In [ ]:
model_version = ModelVersion.V2_0
data, test = pipelines.load_data(
    train_table=model_version.get_table_name(DatasetSplit.TRAIN),
    test_table=model_version.get_table_name(DatasetSplit.TEST),
)

In [ ]:
exp_result_v2_0 = pipelines.run_model(model_version, data, test, TARGET)

  0%|          | 0/10 [00:00<?, ?it/s]

In [7]:
output_path_v2_0 = FIGS_DIR.joinpath(f'{model_version.version_str}/')
output_path_v2_0.mkdir(parents=True, exist_ok=True)
pipelines.save_figures(exp_result_v2_0, path=output_path_v2_0)
pipelines.save_shap_plots(
    exp_result_v2_0, output_path_v2_0, 'shap_top_5', max_display=5
)
pipelines.score_model(exp_result_v2_0)

,accuracy,precision,recall,f1-score,AUC,support
Test,0.927,0.928,0.927,0.927,0.977,25976
Train,0.928,0.929,0.928,0.928,0.977,103904
Test (shuffle),0.561,0.497,0.561,0.404,0.323,25976
Train (shuffle),0.568,0.626,0.568,0.413,0.547,103904


<p align="center">
    <img src="../figs/02/v2.0/shap_top_5_beeswarm.png" width="750px" alt="beeswarm_top_5_v2.0">
    <br>
    <em>Beeswarm SHAP values for the top 5 features of model v2.0.</em>
</p>

<div align="center">

| <center>Dataset</center> | Accuracy | Precision | Recall | F1-Score | ROC AUC |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Test | 0.927 | 0.928 | 0.927 | 0.927 | 0.977 |
| Train | 0.928 | 0.929 | 0.928 | 0.928 | 0.977 |
| Test (Shuffled) | 0.561 | 0.497 | 0.561 | 0.404 | 0.323 |
| Train (Shuffled) | 0.568 | 0.626 | 0.568 | 0.413 | 0.547 |

</div>

Observations:

1. The business class feature worked well and showed a clear division between its two options.

2. The AUC of the target-shuffled train set moved closer to 0.5.

3. The `convenient_time` feature had the least impact on the model and had no clear color gradient in the beeswarm plot so it was dropped in the next model.

4. Many of the lower-impact features had reasonable color gradients, so these were grouped into two new features: `cabin_rating` and `service_rating`. The average rating of `seat_comfort`, `inflight_entertainment`, `leg_room`, and `cleanliness` was used for `cabin_rating` and the average rating of `checkin_service`, `baggage_handling`, `onboard_service`, and `inflight_service` was used for `service_rating`.

# Model v2.1

Drop and/or merge low SHAP features.

Removed: `baggage_handling`, `checkin_service`, `cleanliness`, `convenient_time`, `inflight_entertainment`, `inflight_service`, `leg_room`, `onboard_service`, `seat_comfort`

Added: `cabin_rating`, `service_rating`

In [ ]:
model_version = ModelVersion.V2_1
data, test = pipelines.load_data(
    train_table=model_version.get_table_name(DatasetSplit.TRAIN),
    test_table=model_version.get_table_name(DatasetSplit.TEST),
)

In [ ]:
exp_result_v2_1 = pipelines.run_model(model_version, data, test, TARGET)

  0%|          | 0/10 [00:00<?, ?it/s]

In [11]:
output_path_v2_1 = FIGS_DIR.joinpath(f'{model_version.version_str}/')
output_path_v2_1.mkdir(parents=True, exist_ok=True)
pipelines.save_figures(exp_result_v2_1, path=output_path_v2_1)
pipelines.score_model(exp_result_v2_1)

,accuracy,precision,recall,f1-score,AUC,support
Test,0.911,0.911,0.911,0.910,0.965,25976
Train,0.911,0.912,0.911,0.911,0.965,103904
Test (shuffle),0.561,0.394,0.561,0.403,0.450,25976
Train (shuffle),0.567,0.635,0.567,0.411,0.526,103904


<p align="center">
    <img src="../figs/02/v2.1/shap_bar.png" width="750px" alt="bar_v2.1">
    <br>
    <em>Mean absolute SHAP values for all features of model v2.1.</em>
    <br></br>
    <img src="../figs/02/v2.1/shap_beeswarm.png" width="750px" alt="beeswarm_v2.1">
    <br>
    <em>Beeswarm SHAP values for all features of model v2.1.</em>
</p>

<div align="center">

| <center>Dataset</center> | Accuracy | Precision | Recall | F1-Score | ROC AUC |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Test | 0.911 | 0.911 | 0.911 | 0.910 | 0.965 |
| Train | 0.911 | 0.912 | 0.911 | 0.911 | 0.965 |
| Test (Shuffled) | 0.561 | 0.395 | 0.561 | 0.403 | 0.450 |
| Train (Shuffled) | 0.567 | 0.635 | 0.567 | 0.411 | 0.526 |

</div>

Observations:

1. The `cabin_rating` and `service_rating` features worked well with clear distinctions between low rating impacting the model negatively and high ratings impacting the model positively.

2. The `arrival_delay` feature was a significant step below the other features in model contribution and was removed for the final model.